## Topic 1: Why ordinary linear regression can fail with too many features

## Topic 2: Lasso regression: shrinking some coefficients to exactly zero

## Topic 3: Ridge regression: shrinking coefficients without deleting them

## Topic 4: Choosing lambda/alpha with cross-validation

## Topic 5: Collinearity, VIF, and coefficient instability

## Topic 6: Forward and backward feature selection

## Topic 7: PCA and principal component regression

## Topic 8: When PCA helps — and when it can hurt prediction

# Linear regression fit to all features

## The situation:

## This is linear regression - which you are already familiar with

In [ ]:
Y = 2 * X1 + noise

X1 X2 Y noise
1.  1 3.  1
2.  1. 4.  0

beta1 * X1 + beta2 * X2 = Y
beta1 * 1 + beta2 * 1 = 3
beta1 * 2 + beta2 * 1 = 4
beta1 = 1
beta2 = 2

wanted: beta1 = 2, beta2 = 0a


In [ ]:
Y = 2 * X1 + noise

X1 X2 Y  noise
1. 1  3   1
2. 1  4   0

beta1 * 1 + beta2 * 1 = 3
beta1 * 2 + beta2 * 1 = 4
beta1 = 1  # estimated
beta2 = 2  # estimate
beta1 = 2  # true
beta2 = 0   # true

In [21]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Reproducible example
np.random.seed(42)

# Create 10 rows and 10 columns of features
n_rows = 10
n_features = 10
X = np.random.randn(n_rows, n_features)

# Put into a DataFrame with named columns
feature_names = [f"x{i+1}" for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)

# Construct target Y using only 2 of the 10 columns, with NO intercept
# Add a little noise so the fit is not exact
noise = np.random.normal(loc=0.0, scale=1, size=n_rows)
df["y"] = 3.0 * df["x3"] - 2.5 * df["x8"] + noise

# Fit linear regression using all 10 features, no intercept
X_all = df[feature_names]
y = df["y"]

model = sm.OLS(y, X_all).fit()

# Print coefficients, highlighting the two "truly" nonzero ones
true_nonzero = {"x3", "x8"}

print("Estimated coefficients:")
for name, coef in model.params.items():
    if name in true_nonzero:
        print(f"*** {name}: {coef: .6f}  <-- truly nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nModel summary:")
print(model.summary())

Estimated coefficients:
    x1: -0.510060
    x2:  0.807853
*** x3:  3.106589  <-- truly nonzero
    x4: -0.505276
    x5: -0.142868
    x6:  0.620892
    x7: -0.239256
*** x8: -2.015331  <-- truly nonzero
    x9:  1.018342
    x10:  0.453563

Model summary:
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Thu, 17 Sep 2026   Prob (F-statistic):                nan
Time:                        12:46:57   Log-Likelihood:                 318.81
No. Observations:                  10   AIC:                            -617.6
Df Residuals:                       0   BIC:                            -614.6
Df Model:                           9                                         
Covariance Type:            no

/Users/vonkorff/jupyter-classic/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/Users/vonkorff/jupyter-classic/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/Users/vonkorff/jupyter-classic/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


## Prerequisite refresher: What ordinary least squares is doing

## Ordinary least squares chooses coefficients that minimize squared prediction errors on the training data. But when the model has many features relative to the number of rows, it can fit noise instead of signal.

## Prerequisite refresher: Why scaling matters

## Ridge and lasso penalize coefficient size. If one feature is measured in dollars and another in millions of dollars, the coefficient sizes are not directly comparable. Therefore, scaling is not cosmetic; it changes the penalty.

In [ ]:
Y = X1 + 12 * X2

X1 = inches
X2 = feet
12 * X2 = inches
X2* = 12 * X2

Y = X1 + X2*

MSE + lambda * (1 + 12)  # Lasso
MSE + lambda * (1**2 + 12**2)  # Ridge

In [ ]:
Y = X1 + 1000000 * X2

X2* = 1000000 * X2

Y = X1 + X2*

MSE + lambda * (1 + 1)

## Case study: marketing attribution

## A company has 500 possible customer behavior features: page views, email clicks, coupon usage, time since last purchase, app sessions, and product categories viewed. The business team wants a short list of predictors they can explain. Lasso can help produce a sparse model.

# Lasso regression

## The situation:

## You want to make some coefficients vanish completely.  Maybe your client wants a simple model that they can understand?

## Why this matters: In real data science projects, clients often provide hundreds or thousands of possible predictors. Ordinary regression may fit noise. Lasso is one way to force a simpler model by setting some coefficients exactly to zero.

# Lasso vs. Ridge loss terms

In [27]:
def print_loss_functions():
    print("1) Regular linear regression (two features: x1, x2, no intercept)")
    print("   The usual loss is the sum of squared errors (SSE):")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("   In words: choose the two coefficients to minimize squared prediction error.")
    print()

    print("2) Lasso regression (no intercept)")
    print("   Lasso adds an L1 penalty with lambda:")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("                    + lambda * (|beta1| + |beta2|)")
    print("   In words: minimize squared prediction error, but also penalize the absolute size")
    print("   of the coefficients. This can force some coefficients exactly to zero.")
    print()

    print("3) Ridge regression (no intercept)")
    print("   Ridge adds an L2 penalty with lambda:")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("                    + lambda * (beta1^2 + beta2^2)")
    print("   In words: minimize squared prediction error, but also penalize large coefficients.")
    print("   This shrinks coefficients toward zero, but usually does not make them exactly zero.")
    print()

    print("Notes:")
    print("- lambda controls the strength of regularization.")
    print("- When lambda = 0, lasso and ridge both reduce to ordinary least squares.")
    print("- Since there is no intercept here, all three formulas use only beta1 and beta2.")

print_loss_functions()

1) Regular linear regression (two features: x1, x2, no intercept)
   The usual loss is the sum of squared errors (SSE):
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
   In words: choose the two coefficients to minimize squared prediction error.

2) Lasso regression (no intercept)
   Lasso adds an L1 penalty with lambda:
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
                    + lambda * (|beta1| + |beta2|)
   In words: minimize squared prediction error, but also penalize the absolute size
   of the coefficients. This can force some coefficients exactly to zero.

3) Ridge regression (no intercept)
   Ridge adds an L2 penalty with lambda:
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
                    + lambda * (beta1^2 + beta2^2)
   In words: minimize squared prediction error, but also penalize large coefficients.
   This shrinks coefficients toward zero, but usually does not make them exactly zero.

Notes:
- lambda controls

# Lasso regression example

In [28]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Reproducible example
np.random.seed(42)

# Create 10 rows and 10 columns of features
n_rows = 100
n_features = 10
X = np.random.randn(n_rows, n_features)

# Put into a DataFrame with named columns
feature_names = [f"x{i+1}" for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)

# Construct target Y using only 2 of the 10 columns, with NO intercept
noise = np.random.normal(loc=0.0, scale=1, size=n_rows)
df["y"] = 3.0 * df["x3"] - 2.5 * df["x8"] + noise

X_all = df[feature_names].values
y = df["y"].values

# Lasso is sensitive to feature scale, so standardize features first
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_all)

# "Lambda" in many texts corresponds to alpha in sklearn's Lasso
lambda_param = 0.2

lasso = Lasso(alpha=lambda_param, fit_intercept=False, max_iter=10000)
lasso.fit(X_scaled, y)

print(f"Lasso coefficients with lambda = {lambda_param}:")
for name, coef in zip(feature_names, lasso.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}  <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(lasso.coef_) > 1e-8))

Lasso coefficients with lambda = 0.2:
    x1: -0.000000
    x2: -0.000000
*** x3:  2.805725  <-- nonzero
    x4:  0.000000
    x5: -0.000000
    x6:  0.000000
    x7: -0.000000
*** x8: -2.074801  <-- nonzero
    x9:  0.000000
    x10: -0.000000

Number of nonzero coefficients: 2


In [ ]:
1 * X1 + 1 * X2 -> Lasso: 1 + 1 = 2, Ridge: 1**2 + 1**2 = 2
2 * X1 + 0 * X2 -> Lasso: 2 + 0 = 2, Ridge: 2**2 + 0**2 = 4

# Ridge regression

## The situation:

## You want to make all coefficients have some weight.  Maybe there is collinearity, and you don't want to have to choose between two collinear features?

## Why this matters: Sometimes many variables measure almost the same thing. Ridge lets the model keep related variables while preventing any one coefficient from becoming wildly large.

In [ ]:
X1 = X2
X1 + X2 = 2 * X1 = 2 * X2 = 10 * X1 - 9 * X2
1 + 1, 4 + 0, 0 + 4, 100 + 81

In [29]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Reproducible example
np.random.seed(42)

# Create 10 rows and 10 columns of features
n_rows = 100
n_features = 10
X = np.random.randn(n_rows, n_features)

# Put into a DataFrame with named columns
feature_names = [f"x{i+1}" for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)

# Construct target Y using only 2 of the 10 columns, with NO intercept
noise = np.random.normal(loc=0.0, scale=0.2, size=n_rows)
df["y"] = 3.0 * df["x3"] - 2.5 * df["x8"] + noise

X_all = df[feature_names].values
y = df["y"].values

# Ridge is also sensitive to scale, so standardize features first
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_all)

# "Lambda" in many texts corresponds to alpha in sklearn's Ridge
lambda_param = 5.0

ridge = Ridge(alpha=lambda_param, fit_intercept=False)
ridge.fit(X_scaled, y)

print(f"Ridge coefficients with lambda = {lambda_param}:")
for name, coef in zip(feature_names, ridge.coef_):
    if name in {"x3", "x8"}:
        print(f"*** {name}: {coef: .6f}  <-- truly nonzero in data-generating process")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNote: ridge usually keeps all coefficients nonzero,")
print("but shrinks the coefficients of less important features toward 0.")

Ridge coefficients with lambda = 5.0:
    x1: -0.007542
    x2: -0.003917
*** x3:  2.908065  <-- truly nonzero in data-generating process
    x4:  0.020220
    x5: -0.008383
    x6:  0.000458
    x7: -0.040353
*** x8: -2.144726  <-- truly nonzero in data-generating process
    x9: -0.008755
    x10: -0.007509

Note: ridge usually keeps all coefficients nonzero,
but shrinks the coefficients of less important features toward 0.


In [ ]:
X1 X2 Y  noise   predicted Y   Error
1. 1  3   1         3           0 + 1 * (|2|**2 + |1|**2) = 5
2. 1  4   0         5           1 + 1 * (|0|**2 + |3|**2) = 10
beta1 = 2, beta2 = 1
beta1 = 0, beta2 = 3

In [ ]:
# Scaling
Y = X1 + 1000000 * X2 + noise
X2* = X2 * 1000000
Y = X1 + X2* + noise

In [ ]:
X1 = 1000000, X2 = 1
Equation: Y = X1 + 1000000 * X2 + noise
Lasso or ridge don't work on this
standard scaler
X1* = X1 / 1000000 = 1, X2 = 1
Equation: Y = 1000000 * X1* + 1000000 * X2 + noise

## Case study: housing prices

## Square footage, number of rooms, number of bathrooms, lot size, and neighborhood income are often correlated. Ordinary regression may produce unstable coefficients. Ridge can improve prediction by shrinking correlated predictors together.

# Ridge regression with various values of lambda (or alpha)

## The situation:

## You don't know how strong the lasso / ridge part of the loss should be: you have to test multiple possibilities.

In [11]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Reproducible example
np.random.seed(42)

# Create 10 rows and 10 columns of features
n_rows = 10
n_features = 10
X = np.random.randn(n_rows, n_features)

# Put into a DataFrame with named columns
feature_names = [f"x{i+1}" for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)

# Construct target y using only 2 of the 10 columns, with NO intercept
noise = np.random.normal(loc=0.0, scale=0.2, size=n_rows)
df["y"] = 3.0 * df["x3"] - 2.5 * df["x8"] + noise

X_all = df[feature_names].values
y = df["y"].values

# Ridge is sensitive to scale, so use a pipeline:
# StandardScaler -> Ridge
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("ridge", Ridge(fit_intercept=False))
])

# Lambda values that grow/shrink exponentially
lambda_grid = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]

# With only 10 rows, use a small CV split
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid={"ridge__alpha": lambda_grid},
    scoring="neg_mean_squared_error",
    cv=cv,
    return_train_score=True
)

grid.fit(X_all, y)

print("Best lambda (alpha) found by GridSearchCV:")
print(grid.best_params_["ridge__alpha"])
print(f"Best CV score (negative MSE): {grid.best_score_:.6f}")

# Use the best model
best_model = grid.best_estimator_
best_ridge = best_model.named_steps["ridge"]
best_lambda = best_model.named_steps["ridge"].alpha

print(f"\nRidge coefficients with best lambda = {best_lambda}:")
for name, coef in zip(feature_names, best_ridge.coef_):
    if name in {"x3", "x8"}:
        print(f"*** {name}: {coef: .6f}  <-- truly nonzero in data-generating process")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNote: ridge usually keeps all coefficients nonzero,")
print("but shrinks the coefficients of less important features toward 0.")

# Optional: show all tested lambda values and mean CV scores
print("\nGrid search results:")
for alpha, mean_score in zip(grid.cv_results_["param_ridge__alpha"], grid.cv_results_["mean_test_score"]):
    print(f"lambda = {alpha}: mean CV neg MSE = {mean_score:.6f}")

Best lambda (alpha) found by GridSearchCV:
0.3
Best CV score (negative MSE): -2.719253

Ridge coefficients with best lambda = 0.3:
    x1: -0.203972
    x2:  0.511466
*** x3:  1.890258  <-- truly nonzero in data-generating process
    x4:  0.147061
    x5: -0.096262
    x6:  0.145617
    x7: -0.374164
*** x8: -1.358477  <-- truly nonzero in data-generating process
    x9: -0.315285
    x10:  0.372379

Note: ridge usually keeps all coefficients nonzero,
but shrinks the coefficients of less important features toward 0.

Grid search results:
lambda = 0.01: mean CV neg MSE = -2.850600
lambda = 0.03: mean CV neg MSE = -2.828578
lambda = 0.1: mean CV neg MSE = -2.772693
lambda = 0.3: mean CV neg MSE = -2.719253
lambda = 1.0: mean CV neg MSE = -2.865852
lambda = 3.0: mean CV neg MSE = -3.395203
lambda = 10.0: mean CV neg MSE = -4.471078
lambda = 30.0: mean CV neg MSE = -5.880639


In [2]:
# Interactive demo: alpha/lambda, number of rows, and number of features
# Place this after the Lasso and Ridge sections.

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatLogSlider, IntSlider, fixed
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# Interactive plotting function
# ------------------------------------------------------------
def plot_regularization_demo(
    alpha=0.1,
    n_rows=40,
    n_features=60,
    noise_sd=3.0,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    # --------------------------------------------------------
    # 1) Create synthetic data
    # --------------------------------------------------------
    X = rng.normal(size=(n_rows, n_features))
    feature_names = [f"x{i+1}" for i in range(n_features)]

    # Use only a few truly nonzero features
    true_beta = np.zeros(n_features)

    true_indices = [2, 7, 15]
    true_values = [3.0, -2.5, 2.0]

    for idx, value in zip(true_indices, true_values):
        if idx < n_features:
            true_beta[idx] = value

    noise = rng.normal(loc=0.0, scale=noise_sd, size=n_rows)
    y = X @ true_beta + noise

    # --------------------------------------------------------
    # 2) Train/test split
    # --------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.35, random_state=random_seed
    )

    # --------------------------------------------------------
    # 3) Standardize features
    # --------------------------------------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------
    # 4) Fit current Ridge and Lasso models
    # --------------------------------------------------------
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)

    lasso = Lasso(alpha=alpha, max_iter=20000)
    lasso.fit(X_train_scaled, y_train)

    ridge_train = mean_squared_error(y_train, ridge.predict(X_train_scaled))
    ridge_test = mean_squared_error(y_test, ridge.predict(X_test_scaled))

    lasso_train = mean_squared_error(y_train, lasso.predict(X_train_scaled))
    lasso_test = mean_squared_error(y_test, lasso.predict(X_test_scaled))

    ridge_nonzero = np.sum(np.abs(ridge.coef_) > 1e-8)
    lasso_nonzero = np.sum(np.abs(lasso.coef_) > 1e-8)

    # --------------------------------------------------------
    # 5) Precompute MSE curves over alpha values
    # --------------------------------------------------------
    alpha_grid = np.logspace(-4, 3, 100)

    ridge_train_mse = []
    ridge_test_mse = []
    lasso_train_mse = []
    lasso_test_mse = []
    lasso_nonzero_counts = []

    for a in alpha_grid:
        ridge_a = Ridge(alpha=a)
        ridge_a.fit(X_train_scaled, y_train)

        lasso_a = Lasso(alpha=a, max_iter=20000)
        lasso_a.fit(X_train_scaled, y_train)

        ridge_train_mse.append(mean_squared_error(y_train, ridge_a.predict(X_train_scaled)))
        ridge_test_mse.append(mean_squared_error(y_test, ridge_a.predict(X_test_scaled)))

        lasso_train_mse.append(mean_squared_error(y_train, lasso_a.predict(X_train_scaled)))
        lasso_test_mse.append(mean_squared_error(y_test, lasso_a.predict(X_test_scaled)))

        lasso_nonzero_counts.append(np.sum(np.abs(lasso_a.coef_) > 1e-8))

    ridge_train_mse = np.array(ridge_train_mse)
    ridge_test_mse = np.array(ridge_test_mse)
    lasso_train_mse = np.array(lasso_train_mse)
    lasso_test_mse = np.array(lasso_test_mse)
    lasso_nonzero_counts = np.array(lasso_nonzero_counts)

    best_ridge_alpha = alpha_grid[np.argmin(ridge_test_mse)]
    best_lasso_alpha = alpha_grid[np.argmin(lasso_test_mse)]

    # --------------------------------------------------------
    # 6) Plot only the first 20 coefficients if many features
    # --------------------------------------------------------
    max_display_features = min(n_features, 20)
    display_names = feature_names[:max_display_features]
    display_positions = np.arange(max_display_features)

    fig, axes = plt.subplots(2, 2, figsize=(16, 9))

    # --------------------------------------------------------
    # Plot 1: Ridge coefficients
    # --------------------------------------------------------
    ridge_colors = ["steelblue"] * max_display_features
    for idx in true_indices:
        if idx < max_display_features:
            ridge_colors[idx] = "orange"

    axes[0, 0].bar(display_positions, ridge.coef_[:max_display_features], color=ridge_colors)
    axes[0, 0].axhline(0, color="black", linewidth=1)
    axes[0, 0].set_title(f"Ridge coefficients, alpha = {alpha:.4f}")
    axes[0, 0].set_ylabel("Coefficient value")
    axes[0, 0].set_xticks(display_positions)
    axes[0, 0].set_xticklabels(display_names, rotation=45)

    # --------------------------------------------------------
    # Plot 2: Lasso coefficients
    # --------------------------------------------------------
    lasso_colors = ["seagreen"] * max_display_features
    for idx in true_indices:
        if idx < max_display_features:
            lasso_colors[idx] = "orange"

    axes[0, 1].bar(display_positions, lasso.coef_[:max_display_features], color=lasso_colors)
    axes[0, 1].axhline(0, color="black", linewidth=1)
    axes[0, 1].set_title(
        f"Lasso coefficients, alpha = {alpha:.4f}\n"
        f"Nonzero coefficients: {lasso_nonzero}"
    )
    axes[0, 1].set_ylabel("Coefficient value")
    axes[0, 1].set_xticks(display_positions)
    axes[0, 1].set_xticklabels(display_names, rotation=45)

    # --------------------------------------------------------
    # Plot 3: Train/test MSE curves
    # --------------------------------------------------------
    axes[1, 0].plot(alpha_grid, ridge_train_mse, label="Ridge train MSE", color="blue", linestyle="--")
    axes[1, 0].plot(alpha_grid, ridge_test_mse, label="Ridge test MSE", color="blue")
    axes[1, 0].plot(alpha_grid, lasso_train_mse, label="Lasso train MSE", color="green", linestyle="--")
    axes[1, 0].plot(alpha_grid, lasso_test_mse, label="Lasso test MSE", color="green")
    axes[1, 0].axvline(alpha, color="red", linestyle=":", linewidth=2, label="current alpha")
    axes[1, 0].axvline(best_ridge_alpha, color="blue", linestyle=":", alpha=0.5, label="best ridge test alpha")
    axes[1, 0].axvline(best_lasso_alpha, color="green", linestyle=":", alpha=0.5, label="best lasso test alpha")
    axes[1, 0].set_xscale("log")
    axes[1, 0].set_xlabel("alpha / lambda")
    axes[1, 0].set_ylabel("Mean squared error")
    axes[1, 0].set_title("Train/test MSE as regularization changes")
    axes[1, 0].legend(fontsize=8)

    # --------------------------------------------------------
    # Plot 4: Number of nonzero lasso coefficients
    # --------------------------------------------------------
    axes[1, 1].plot(alpha_grid, lasso_nonzero_counts, marker="o", markersize=3, color="purple")
    axes[1, 1].axvline(alpha, color="red", linestyle=":", linewidth=2)
    axes[1, 1].set_xscale("log")
    axes[1, 1].set_xlabel("alpha / lambda")
    axes[1, 1].set_ylabel("Number of nonzero coefficients")
    axes[1, 1].set_title("Lasso feature selection effect")

    fig.suptitle(
        "Interactive Regularization Demo: Ridge shrinks, Lasso can delete",
        fontsize=16,
        y=1.02
    )

    plt.tight_layout()
    plt.show()

    print(f"Rows: {n_rows}")
    print(f"Features: {n_features}")
    print(f"Training rows: {len(y_train)}")
    print(f"Test rows: {len(y_test)}")
    print(f"Noise standard deviation: {noise_sd}")
    print()
    print(f"Current alpha: {alpha:.4f}")
    print()
    print("Ridge:")
    print(f"  Train MSE: {ridge_train:.4f}")
    print(f"  Test  MSE: {ridge_test:.4f}")
    print(f"  Number of nonzero coefficients: {ridge_nonzero}")
    print(f"  Best test alpha in grid: {best_ridge_alpha:.4f}")
    print()
    print("Lasso:")
    print(f"  Train MSE: {lasso_train:.4f}")
    print(f"  Test  MSE: {lasso_test:.4f}")
    print(f"  Number of nonzero coefficients: {lasso_nonzero}")
    print(f"  Best test alpha in grid: {best_lasso_alpha:.4f}")
    print()
    print("Orange bars are truly nonzero features, if they are among the first 20 displayed.")

# ------------------------------------------------------------
# Slider controls
# ------------------------------------------------------------
interact(
    plot_regularization_demo,
    alpha=FloatLogSlider(
        value=0.3,
        base=10,
        min=-4,
        max=3,
        step=0.05,
        description="alpha",
        continuous_update=False
    ),
    n_rows=IntSlider(
        value=40,
        min=20,
        max=300,
        step=10,
        description="rows",
        continuous_update=False
    ),
    n_features=IntSlider(
        value=60,
        min=5,
        max=200,
        step=5,
        description="features",
        continuous_update=False
    ),
    noise_sd=FloatLogSlider(
        value=3.0,
        base=10,
        min=-1,
        max=1,
        step=0.05,
        description="noise",
        continuous_update=False
    ),
    random_seed=fixed(42)
)

interactive(children=(FloatLogSlider(value=0.3, continuous_update=False, description='alpha', max=3.0, min=-4.…

<function __main__.plot_regularization_demo(alpha=0.1, n_rows=40, n_features=60, noise_sd=3.0, random_seed=42)>

# Ordinary, lasso, and ridge regression with collinearity and VIF

## The situation:

## This time, there is collinearity in the data.  What happens with each regression type?

## Why this matters:

## Real datasets often contain predictors that overlap heavily, making individual regression coefficients unstable or misleading even when predictions are decent. This matters because we may not know which feature truly deserves credit.

In [21]:
#import matplotlib.pyplot as plt
#plt.plot(x1, x2)

In [20]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

# Reproducible example
np.random.seed(42)

# Create 10 rows with two strongly collinear columns
n_rows = 10
x1 = np.random.randn(n_rows)

# Make x2 almost a copy of x1, which creates strong multicollinearity
x2 = x1 + np.random.normal(loc=0.0, scale=0.01, size=n_rows)

# Target is a linear sum of the two columns plus noise
noise = np.random.normal(loc=0.0, scale=0.2, size=n_rows)
coef_x1 = 2.0
coef_x2 = -3.0
y = coef_x1 * x1 + coef_x2 * x2 + noise

print("x1 coefficient:", coef_x1)
print("x2 coefficient:", coef_x2)

df = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "y": y
})

print("Data:")
print(df)
print()

# -------------------------
# Show collinearity with VIF
# -------------------------
X = df[["x1", "x2"]]

vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("VIF values:")
print(vif_df)
print()

# -------------------------
# 1) Regular regression
# -------------------------
ols_model = sm.OLS(df["y"], X).fit()

print("Regular regression (OLS) coefficients:")
for name, coef in ols_model.params.items():
    print(f"{name}: {coef: .6f}")
print()

# -------------------------
# 2) Lasso regression
# -------------------------
# Standardize features for lasso
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X.values)

lasso_lambda = 0.02
lasso = Lasso(alpha=lasso_lambda, fit_intercept=False, max_iter=10000)
lasso.fit(X_scaled, df["y"].values)

print(f"Lasso coefficients (lambda = {lasso_lambda}):")
for name, coef in zip(X.columns, lasso.coef_):
    print(f"{name}: {coef: .6f}")
print()

# -------------------------
# 3) Ridge regression
# -------------------------
ridge_lambda = 0.5
ridge = Ridge(alpha=ridge_lambda, fit_intercept=False)
ridge.fit(X_scaled, df["y"].values)

print(f"Ridge coefficients (lambda = {ridge_lambda}):")
for name, coef in zip(X.columns, ridge.coef_):
    print(f"{name}: {coef: .6f}")
print()

#2 * x1 - 3 * x2
#2 * x1 - 3 * x1
#= - x1
#= - x2
#= 1000 * x1 - 1001 * x2

x1 coefficient: 2.0
x2 coefficient: -3.0
Data:
         x1        x2         y
0  0.496714  0.492080 -0.189682
1 -0.138264 -0.142922  0.107081
2  0.647689  0.650108 -0.641442
3  1.523030  1.503897 -1.750581
4 -0.234153 -0.251403  0.177024
5 -0.234137 -0.239760  0.273190
6  1.579213  1.569085 -1.779027
7  0.767435  0.770577 -0.701723
8 -0.469474 -0.478555  0.376587
9  0.542560  0.528437 -0.558530

VIF values:
  feature          VIF
0      x1  7452.161933
1      x2  7452.161933

Regular regression (OLS) coefficients:
x1: -6.255476
x2:  5.225959

Lasso coefficients (lambda = 0.02):
x1: -0.712432
x2: -0.000000

Ridge coefficients (lambda = 0.5):
x1: -0.365011
x2: -0.349189



In [ ]:
X1 = 1, X2 = 1
X1 + X2, 10 X1 - 8 X2

In [26]:
x1 = 1
x2 = 1
-6.2554 * x1 + 5.2259 * x2, 2 * x1 - 3 * x2

(-1.0294999999999996, -1)

In [4]:
# Interactive demo: collinearity, VIF, OLS instability, Ridge stabilization, and Lasso selection
# Fixed version:
#   Ridge and Lasso are fit on standardized X, but their coefficients are converted
#   back to the original feature scale before plotting against the true coefficients.

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, fixed
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor


def fit_scaled_model_and_convert_back(model, X, y):
    """
    Fit a model on standardized X, then convert coefficients back to original X units.

    If X_scaled_j = (X_j - mean_j) / scale_j, then:

        y_hat = intercept_scaled + sum_j coef_scaled_j * X_scaled_j

    becomes:

        y_hat = intercept_original + sum_j coef_original_j * X_j

    where:

        coef_original_j = coef_scaled_j / scale_j
        intercept_original = intercept_scaled - sum_j coef_scaled_j * mean_j / scale_j
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model.fit(X_scaled, y)

    coef_original = model.coef_ / scaler.scale_
    intercept_original = model.intercept_ - np.sum(model.coef_ * scaler.mean_ / scaler.scale_)

    y_pred = intercept_original + X @ coef_original
    mse = mean_squared_error(y, y_pred)

    return coef_original, intercept_original, mse


def plot_collinearity_demo(
    collinearity_noise=1.0,
    y_noise=0.05,
    ridge_alpha=1.0,
    lasso_alpha=0.05,
    n_rows=80,
    beta1=2.0,
    beta2=-3.0,
    random_seed=42
):
    rng = np.random.default_rng(random_seed)

    # ------------------------------------------------------------
    # 1) Create two related features
    # ------------------------------------------------------------
    x1 = rng.normal(0, 1, size=n_rows)
    x2 = x1 + rng.normal(0, collinearity_noise, size=n_rows)

    y = beta1 * x1 + beta2 * x2 + rng.normal(0, y_noise, size=n_rows)

    X = np.column_stack([x1, x2])
    feature_names = ["x1", "x2"]

    # ------------------------------------------------------------
    # 2) Correlation and VIF
    # ------------------------------------------------------------
    corr = np.corrcoef(x1, x2)[0, 1]

    try:
        vif_x1 = variance_inflation_factor(X, 0)
        vif_x2 = variance_inflation_factor(X, 1)
    except Exception:
        vif_x1 = np.inf
        vif_x2 = np.inf

    # ------------------------------------------------------------
    # 3) Fit OLS on original scale
    # ------------------------------------------------------------
    ols = LinearRegression(fit_intercept=True)
    ols.fit(X, y)

    ols_coef = ols.coef_
    ols_intercept = ols.intercept_
    ols_mse = mean_squared_error(y, ols.predict(X))

    # ------------------------------------------------------------
    # 4) Fit Ridge and Lasso on standardized features,
    #    then convert coefficients back to original feature scale.
    # ------------------------------------------------------------
    ridge_coef, ridge_intercept, ridge_mse = fit_scaled_model_and_convert_back(
        Ridge(alpha=ridge_alpha, fit_intercept=True),
        X,
        y
    )

    lasso_coef, lasso_intercept, lasso_mse = fit_scaled_model_and_convert_back(
        Lasso(alpha=lasso_alpha, fit_intercept=True, max_iter=20000),
        X,
        y
    )

    # ------------------------------------------------------------
    # 5) Repeated simulations to show coefficient instability
    # ------------------------------------------------------------
    n_sims = 200

    ols_coefs = []
    ridge_coefs = []
    lasso_coefs = []

    for sim in range(n_sims):
        sim_rng = np.random.default_rng(random_seed + sim + 1)

        x1_sim = sim_rng.normal(0, 1, size=n_rows)
        x2_sim = x1_sim + sim_rng.normal(0, collinearity_noise, size=n_rows)
        y_sim = beta1 * x1_sim + beta2 * x2_sim + sim_rng.normal(0, y_noise, size=n_rows)

        X_sim = np.column_stack([x1_sim, x2_sim])

        ols_sim = LinearRegression(fit_intercept=True)
        ols_sim.fit(X_sim, y_sim)
        ols_coefs.append(ols_sim.coef_)

        ridge_sim_coef, _, _ = fit_scaled_model_and_convert_back(
            Ridge(alpha=ridge_alpha, fit_intercept=True),
            X_sim,
            y_sim
        )
        ridge_coefs.append(ridge_sim_coef)

        lasso_sim_coef, _, _ = fit_scaled_model_and_convert_back(
            Lasso(alpha=lasso_alpha, fit_intercept=True, max_iter=20000),
            X_sim,
            y_sim
        )
        lasso_coefs.append(lasso_sim_coef)

    ols_coefs = np.array(ols_coefs)
    ridge_coefs = np.array(ridge_coefs)
    lasso_coefs = np.array(lasso_coefs)

    # ------------------------------------------------------------
    # 6) Plot
    # ------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))

    # ------------------------------------------------------------
    # Plot 1: x1 vs x2
    # ------------------------------------------------------------
    axes[0, 0].scatter(x1, x2, alpha=0.75)
    axes[0, 0].set_xlabel("x1")
    axes[0, 0].set_ylabel("x2")
    axes[0, 0].set_title(
        "Collinearity in the features\n"
        f"corr(x1, x2) = {corr:.4f}"
    )

    min_val = min(x1.min(), x2.min())
    max_val = max(x1.max(), x2.max())
    axes[0, 0].plot(
        [min_val, max_val],
        [min_val, max_val],
        color="red",
        linestyle="--",
        linewidth=1
    )

    # ------------------------------------------------------------
    # Plot 2: Coefficients from current dataset
    # ------------------------------------------------------------
    width = 0.25
    positions = np.arange(2)

    axes[0, 1].bar(
        positions - width,
        ols_coef,
        width=width,
        label="OLS",
        color="gray"
    )
    axes[0, 1].bar(
        positions,
        ridge_coef,
        width=width,
        label="Ridge",
        color="steelblue"
    )
    axes[0, 1].bar(
        positions + width,
        lasso_coef,
        width=width,
        label="Lasso",
        color="seagreen"
    )

    axes[0, 1].scatter(
        positions,
        [beta1, beta2],
        color="red",
        marker="*",
        s=180,
        label="True coefficient"
    )

    axes[0, 1].axhline(0, color="black", linewidth=1)
    axes[0, 1].set_xticks(positions)
    axes[0, 1].set_xticklabels(feature_names)
    axes[0, 1].set_ylabel("Estimated coefficient, original feature scale")
    axes[0, 1].set_title("Current fitted coefficients")
    axes[0, 1].legend()

    # ------------------------------------------------------------
    # Plot 3: Coefficient instability across repeated datasets
    # ------------------------------------------------------------
    axes[1, 0].scatter(
        ols_coefs[:, 0],
        ols_coefs[:, 1],
        alpha=0.35,
        label="OLS",
        color="gray"
    )
    axes[1, 0].scatter(
        ridge_coefs[:, 0],
        ridge_coefs[:, 1],
        alpha=0.35,
        label="Ridge",
        color="steelblue"
    )
    axes[1, 0].scatter(
        lasso_coefs[:, 0],
        lasso_coefs[:, 1],
        alpha=0.35,
        label="Lasso",
        color="seagreen"
    )

    axes[1, 0].scatter(
        [beta1],
        [beta2],
        color="red",
        s=120,
        marker="*",
        label="True coefficients"
    )

    axes[1, 0].axhline(0, color="black", linewidth=1)
    axes[1, 0].axvline(0, color="black", linewidth=1)
    axes[1, 0].set_xlabel("Estimated coefficient for x1, original scale")
    axes[1, 0].set_ylabel("Estimated coefficient for x2, original scale")
    axes[1, 0].set_title("Coefficient instability across repeated samples")
    axes[1, 0].legend(fontsize=8)

    # ------------------------------------------------------------
    # Plot 4: VIF and model summary text
    # ------------------------------------------------------------
    axes[1, 1].axis("off")

    lasso_nonzero = np.sum(np.abs(lasso_coef) > 1e-8)

    summary_text = (
        f"Data-generating model:\n"
        f"  y = {beta1:.2f} x1 + {beta2:.2f} x2 + noise\n\n"
        f"Feature relationship:\n"
        f"  x2 = x1 + N(0, {collinearity_noise:.4f})\n"
        f"  corr(x1, x2) = {corr:.4f}\n"
        f"  VIF(x1) = {vif_x1:.2f}\n"
        f"  VIF(x2) = {vif_x2:.2f}\n\n"
        f"Current model MSE:\n"
        f"  OLS   MSE = {ols_mse:.6f}\n"
        f"  Ridge MSE = {ridge_mse:.6f}\n"
        f"  Lasso MSE = {lasso_mse:.6f}\n\n"
        f"Current coefficients, original scale:\n"
        f"  True:  x1 = {beta1: .3f}, x2 = {beta2: .3f}\n"
        f"  OLS:   x1 = {ols_coef[0]: .3f}, x2 = {ols_coef[1]: .3f}\n"
        f"  Ridge: x1 = {ridge_coef[0]: .3f}, x2 = {ridge_coef[1]: .3f}\n"
        f"  Lasso: x1 = {lasso_coef[0]: .3f}, x2 = {lasso_coef[1]: .3f}\n\n"
        f"Lasso nonzero coefficients: {lasso_nonzero}"
    )

    axes[1, 1].text(
        0.0,
        1.0,
        summary_text,
        va="top",
        ha="left",
        fontsize=11,
        family="monospace"
    )

    fig.suptitle(
        "Interactive Collinearity Demo: OLS becomes unstable, Ridge stabilizes, Lasso may choose one",
        fontsize=15,
        y=1.02
    )

    plt.tight_layout()
    plt.show()

    print("Teaching notes:")
    print("- All plotted coefficients are now on the original feature scale.")
    print("- The red star is the true coefficient pair used to generate y.")
    print("- As x2 noise gets smaller, x1 and x2 become more similar.")
    print("- Then correlation and VIF increase.")
    print("- OLS can become unstable because the model cannot reliably assign credit to x1 versus x2.")
    print("- Ridge usually shrinks and stabilizes coefficients, often spreading weight across correlated variables.")
    print("- Lasso often chooses one correlated variable and drops the other.")
    print("- Prediction error can still be reasonable even when individual coefficients are hard to interpret.")


interact(
    plot_collinearity_demo,
    collinearity_noise=FloatLogSlider(
        value=1.0,
        base=10,
        min=-3,
        max=1,
        step=0.05,
        description="x2 noise",
        continuous_update=False
    ),
    y_noise=FloatLogSlider(
        value=0.05,
        base=10,
        min=-3,
        max=1,
        step=0.05,
        description="y noise",
        continuous_update=False
    ),
    ridge_alpha=FloatLogSlider(
        value=1.0,
        base=10,
        min=-3,
        max=2,
        step=0.05,
        description="ridge α",
        continuous_update=False
    ),
    lasso_alpha=FloatLogSlider(
        value=0.05,
        base=10,
        min=-3,
        max=1,
        step=0.05,
        description="lasso α",
        continuous_update=False
    ),
    n_rows=IntSlider(
        value=80,
        min=20,
        max=300,
        step=10,
        description="rows",
        continuous_update=False
    ),
    beta1=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.5,
        description="true β1",
        continuous_update=False
    ),
    beta2=FloatSlider(
        value=-3.0,
        min=-5.0,
        max=5.0,
        step=0.5,
        description="true β2",
        continuous_update=False
    ),
    random_seed=fixed(42)
)

interactive(children=(FloatLogSlider(value=1.0, continuous_update=False, description='x2 noise', max=1.0, min=…

<function __main__.plot_collinearity_demo(collinearity_noise=1.0, y_noise=0.05, ridge_alpha=1.0, lasso_alpha=0.05, n_rows=80, beta1=2.0, beta2=-3.0, random_seed=42)>

In [ ]:
(-6, 5), (2, -3)

# Forward selection

## The situation:

## You want a simpler model that does not use all coefficients, as with lasso.
## Advantages of lasso: it's not biased toward some combinations of parameters over others.  Forward selection tries some combinations and not others.
## Advantages of forward selection: you can see which parameters it added first, in order of importance.

# Why this matters

## Forward selection builds a simpler model by adding one useful feature at a time. This matters when interpretability, data collection cost, or avoiding unnecessary predictors is important.

In [15]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

np.random.seed(42)

# ------------------------------------------------------------
# 1) Create data with 10 features, but only 3 truly matter
# ------------------------------------------------------------
n = 500

X = pd.DataFrame(
    np.random.normal(size=(n, 10)),
    columns=[f"x{i}" for i in range(1, 11)]
)

# True model uses only x2, x5, x9
y = 4 * X["x2"] - 2 * X["x5"] + 3 * X["x9"] + np.random.normal(scale=1.0, size=n)

# ------------------------------------------------------------
# 2) Forward selection using cross-validated MSE
# ------------------------------------------------------------
def forward_selection(X, y, max_features=10):
    remaining = list(X.columns)
    selected = []
    history = []

    for step in range(max_features):
        best_feature = None
        best_score = np.inf

        for feature in remaining:
            candidate_features = selected + [feature]
            model = LinearRegression()
            scores = cross_val_score(
                model,
                X[candidate_features],
                y,
                cv=5,
                scoring="neg_mean_squared_error"
            )
            mse = -scores.mean()

            if mse < best_score:
                best_score = mse
                best_feature = feature

        selected.append(best_feature)
        remaining.remove(best_feature)
        history.append((step + 1, best_feature, best_score))

    return selected, history

selected_features, history = forward_selection(X, y, max_features=10)

print("Forward selection order:")
for step, feature, score in history:
    print(f"Step {step}: add {feature:>3s} | CV MSE = {score:.4f}")

print("\nFinal selected order:")
print(selected_features)

# ------------------------------------------------------------
# 3) Show where the true features were found
# ------------------------------------------------------------
true_features = {"x2", "x5", "x9"}
selected_set = set(selected_features)

print("\nTrue features:", true_features)
#print("Selected features:", selected_set)
#print("True features recovered:", true_features.intersection(selected_set))

# ------------------------------------------------------------
# 4) Compare model size vs performance
# ------------------------------------------------------------
print("\nCV MSE by number of selected features:")
for k in range(1, 11):
    feats = selected_features[:k]
    model = LinearRegression()
    scores = cross_val_score(
        model,
        X[feats],
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
    mse = -scores.mean()
    print(f"{k} features: {mse:.4f}")
    

Forward selection order:
Step 1: add  x2 | CV MSE = 13.4709
Step 2: add  x9 | CV MSE = 4.9094
Step 3: add  x5 | CV MSE = 0.9375
Step 4: add x10 | CV MSE = 0.9380
Step 5: add  x6 | CV MSE = 0.9386
Step 6: add  x8 | CV MSE = 0.9424
Step 7: add  x1 | CV MSE = 0.9459
Step 8: add  x3 | CV MSE = 0.9506
Step 9: add  x7 | CV MSE = 0.9558
Step 10: add  x4 | CV MSE = 0.9639

Final selected order:
['x2', 'x9', 'x5', 'x10', 'x6', 'x8', 'x1', 'x3', 'x7', 'x4']

True features: {'x5', 'x2', 'x9'}

CV MSE by number of selected features:
1 features: 13.4709
2 features: 4.9094
3 features: 0.9375
4 features: 0.9380
5 features: 0.9386
6 features: 0.9424
7 features: 0.9459
8 features: 0.9506
9 features: 0.9558
10 features: 0.9639


# Backward selection

## Like forward selection, but remove parameters instead of adding them.

In [28]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)

n = 500
X = pd.DataFrame(
    np.random.normal(size=(n, 10)),
    columns=[f"x{i}" for i in range(1, 11)]
)
y = 4 * X["x2"] - 2 * X["x5"] + 3 * X["x9"] + np.random.normal(scale=1.0, size=n)

def cv_mse(X, y, features):
    model = LinearRegression()
    scores = cross_val_score(
        model,
        X[features],
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
    return -scores.mean()

def backward_selection(X, y, min_features=1):
    selected = list(X.columns)
    history = []

    while len(selected) > min_features:
        best_score = np.inf
        best_candidate = None
        feature_removed = None

        for feature in selected:
            candidate = [f for f in selected if f != feature]
            mse = cv_mse(X, y, candidate)

            if mse < best_score:
                best_score = mse
                best_candidate = candidate
                feature_removed = feature

        selected = best_candidate
        history.append((feature_removed, selected.copy(), best_score))

    return selected, history

selected_features, history = backward_selection(X, y, min_features=1)

print("Backward selection path:")
print(f"Start with all 10 features | CV MSE = {cv_mse(X, y, list(X.columns)):.4f}")
for removed_feature, remaining_features, score in history:
    print(f"Remove {removed_feature:>3s} -> keep {remaining_features} | CV MSE = {score:.4f}")

#print("\nFinal selected feature(s):", selected_features)

true_features = {"x2", "x5", "x9"}
print("True features:", true_features)
#print("Recovered true features:", true_features.intersection(set(selected_features)))

Backward selection path:
Start with all 10 features | CV MSE = 0.9639
Remove  x4 -> keep ['x1', 'x2', 'x3', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'] | CV MSE = 0.9558
Remove  x7 -> keep ['x1', 'x2', 'x3', 'x5', 'x6', 'x8', 'x9', 'x10'] | CV MSE = 0.9506
Remove  x3 -> keep ['x1', 'x2', 'x5', 'x6', 'x8', 'x9', 'x10'] | CV MSE = 0.9459
Remove  x1 -> keep ['x2', 'x5', 'x6', 'x8', 'x9', 'x10'] | CV MSE = 0.9424
Remove  x8 -> keep ['x2', 'x5', 'x6', 'x9', 'x10'] | CV MSE = 0.9386
Remove  x6 -> keep ['x2', 'x5', 'x9', 'x10'] | CV MSE = 0.9380
Remove x10 -> keep ['x2', 'x5', 'x9'] | CV MSE = 0.9375
Remove  x5 -> keep ['x2', 'x9'] | CV MSE = 4.9094
Remove  x9 -> keep ['x2'] | CV MSE = 13.4709
True features: {'x2', 'x9', 'x5'}


# PCA and PCR where one column varies less and is less important

## Case study: sensor data

## A manufacturer collects hundreds of vibration and temperature readings from machines. Many sensors move together. PCA can compress the sensor readings into a few directions before regression, but PCR can miss low-variance signals if those signals are predictive of failure.

## The situation:

## You want to train on the most important linear combinations of features.
## You think that the most important feature might be Feature1 + Feature2 rather than just Feature1 or Feature2
## This might happen with collinearity (X1 + X2 matters a lot more than X1 - X2).

## Why this matters

## PCA helps reduce many overlapping features into a smaller number of summary directions. This matters because it can simplify high-dimensional data, but it may discard low-variance information that is important for prediction.

In [21]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# -----------------------------
# 1) Make a 3-column dataset
# -----------------------------
print("Creating a 3-column dataset with independent columns...")
print("Goal: X1 and X2 have much larger sigma than X3.")
print("Also: X1, X2, X3 are uncorrelated.")
print("Target: Y = X1 + X2 + X3\n")

rng = np.random.default_rng(42)
n = 5000

sigma1 = 10.0
sigma2 = 8.0
sigma3 = 1.0

X1 = rng.normal(0, sigma1, size=n)
X2 = rng.normal(0, sigma2, size=n)
X3 = rng.normal(0, sigma3, size=n)

X = np.column_stack([X1, X2, X3])
y = X1 + X2 + X3 + np.random.normal(0, 1, n)

print("Empirical standard deviations:")
print("  std(X1) =", X1.std(ddof=1))
print("  std(X2) =", X2.std(ddof=1))
print("  std(X3) =", X3.std(ddof=1))
print()

corr = np.corrcoef(X, rowvar=False)
print("Empirical correlation matrix:")
print(corr)
print()

# -----------------------------
# 2) Run PCA
# -----------------------------
print("Running PCA on X (no standardization, so variance dominates)...")
pca = PCA(n_components=3)
pca.fit(X)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print()

print("Principal component vectors (rows):")
for i, vec in enumerate(pca.components_, start=1):
    print(f"  PC{i}: {vec}")
print()

print("Interpretation:")
print("  If PC1 and PC2 are mostly aligned with X1 and X2,")
print("  their vectors should have large weight in the first two coordinates.")
print()

# -----------------------------
# 3) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# -----------------------------
# 4) PCR: PCA -> regression on top k PCs
# -----------------------------
def run_pcr(k):
    print(f"Running PCR with k = {k} principal components...")
    model = Pipeline([
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print("  Train R^2 :", train_r2)
    print("  Test  R^2 :", test_r2)
    print("  Train RMSE:", train_rmse)
    print("  Test  RMSE:", test_rmse)

    pca_step = model.named_steps["pca"]
    lr_step = model.named_steps["lr"]

    print("  PCA components used by PCR:")
    for i, vec in enumerate(pca_step.components_, start=1):
        print(f"    PC{i}: {vec}")

    print("  Regression coefficients in PC space:")
    print("   ", lr_step.coef_)
    print("  Intercept:")
    print("   ", lr_step.intercept_)
    print()

    return model

print("Now running PCR with different numbers of PCs...")
print("Because the target is Y = X1 + X2 + X3, all three features matter.")
print("But PCA keeps directions of largest variance, not directions most useful for prediction.\n")

model1 = run_pcr(1)
model2 = run_pcr(2)
model3 = run_pcr(3)

# -----------------------------
# 5) Compare to ordinary linear regression
# -----------------------------
print("Running ordinary linear regression on the original columns for comparison...")
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_test_lr = lr.predict(X_test)
print("  Test R^2 :", r2_score(y_test, y_pred_test_lr))
print("  Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test_lr)))
print("  Coefficients on original columns:")
print("   ", lr.coef_)
print("  Intercept:")
print("   ", lr.intercept_)
print()

print("Done.")
print("Expected qualitative result:")
print("  - PCA: PC1 mostly points along X1, PC2 mostly along X2.")
print("  - PCR with only 1 or 2 PCs may miss useful information in X3.")
print("  - PCR with all 3 PCs should recover the target well.")

Creating a 3-column dataset with independent columns...
Goal: X1 and X2 have much larger sigma than X3.
Also: X1, X2, X3 are uncorrelated.
Target: Y = X1 + X2 + X3

Empirical standard deviations:
  std(X1) = 9.994544724124854
  std(X2) = 8.10543551859412
  std(X3) = 0.9995416728220029

Empirical correlation matrix:
[[ 1.         -0.03374026 -0.01696251]
 [-0.03374026  1.          0.0146961 ]
 [-0.01696251  0.0146961   1.        ]]

Running PCA on X (no standardization, so variance dominates)...
Explained variance ratio:
[0.60093347 0.39307216 0.00599437]

Principal component vectors (rows):
  PC1: [ 0.99685796 -0.07918941 -0.00179954]
  PC2: [0.07919236 0.99685802 0.00163255]
  PC3: [ 0.0016646  -0.00176993  0.99999705]

Interpretation:
  If PC1 and PC2 are mostly aligned with X1 and X2,
  their vectors should have large weight in the first two coordinates.

Now running PCR with different numbers of PCs...
Because the target is Y = X1 + X2 + X3, all three features matter.
But PCA keeps

# PCA and PCR where one column varies less but is still important

## The situation:

## PCR doesn't really work very well when the less-varying column (or combination of columns) actually matters more in the regression.

In [17]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# -----------------------------
# 1) Make a 3-column dataset
# -----------------------------
print("Creating a 3-column dataset with independent columns...")
print("Goal: X1 and X2 have much larger sigma than X3.")
print("Also: X1, X2, X3 are uncorrelated.")
print("Target: Y = X1 + X2 + 10 * X3\n")

rng = np.random.default_rng(42)
n = 5000

sigma1 = 10.0
sigma2 = 8.0
sigma3 = 1.0

X1 = rng.normal(0, sigma1, size=n)
X2 = rng.normal(0, sigma2, size=n)
X3 = rng.normal(0, sigma3, size=n)

X = np.column_stack([X1, X2, X3])
y = X1 + X2 + 10 * X3 + np.random.normal(0, 1, n) # *************** NEW *************** 

print("Empirical standard deviations:")
print("  std(X1) =", X1.std(ddof=1))
print("  std(X2) =", X2.std(ddof=1))
print("  std(X3) =", X3.std(ddof=1))
print()

corr = np.corrcoef(X, rowvar=False)
print("Empirical correlation matrix:")
print(corr)
print()

# -----------------------------
# 2) Run PCA
# -----------------------------
print("Running PCA on X (no standardization, so variance dominates)...")
pca = PCA(n_components=3)
pca.fit(X)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print()

print("Principal component vectors (rows):")
for i, vec in enumerate(pca.components_, start=1):
    print(f"  PC{i}: {vec}")
print()

print("Interpretation:")
print("  If PC1 and PC2 are mostly aligned with X1 and X2,")
print("  their vectors should have large weight in the first two coordinates.")
print()

# -----------------------------
# 3) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# -----------------------------
# 4) PCR: PCA -> regression on top k PCs
# -----------------------------
def run_pcr(k):
    print(f"Running PCR with k = {k} principal components...")
    model = Pipeline([
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print("  Train R^2 :", train_r2)
    print("  Test  R^2 :", test_r2)
    print("  Train RMSE:", train_rmse)
    print("  Test  RMSE:", test_rmse)

    pca_step = model.named_steps["pca"]
    lr_step = model.named_steps["lr"]

    print("  PCA components used by PCR:")
    for i, vec in enumerate(pca_step.components_, start=1):
        print(f"    PC{i}: {vec}")

    print("  Regression coefficients in PC space:")
    print("   ", lr_step.coef_)
    print("  Intercept:")
    print("   ", lr_step.intercept_)
    print()

    return model

print("Now running PCR with different numbers of PCs...")
print("Because the target is Y = X1 + X2 + X3, all three features matter.")
print("But PCA keeps directions of largest variance, not directions most useful for prediction.\n")

model1 = run_pcr(1)
model2 = run_pcr(2)
model3 = run_pcr(3)

# -----------------------------
# 5) Compare to ordinary linear regression
# -----------------------------
print("Running ordinary linear regression on the original columns for comparison...")
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_test_lr = lr.predict(X_test)
print("  Test R^2 :", r2_score(y_test, y_pred_test_lr))
print("  Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test_lr)))
print("  Coefficients on original columns:")
print("   ", lr.coef_)
print("  Intercept:")
print("   ", lr.intercept_)
print()

print("Done.")
print("Expected qualitative result:")
print("  - PCA: PC1 mostly points along X1, PC2 mostly along X2.")
print("  - PCR with only 1 or 2 PCs may miss useful information in X3.")
print("  - PCR with all 3 PCs should recover the target well.")

Creating a 3-column dataset with independent columns...
Goal: X1 and X2 have much larger sigma than X3.
Also: X1, X2, X3 are uncorrelated.
Target: Y = X1 + X2 + 10 * X3

Empirical standard deviations:
  std(X1) = 9.994544724124854
  std(X2) = 8.10543551859412
  std(X3) = 0.9995416728220029

Empirical correlation matrix:
[[ 1.         -0.03374026 -0.01696251]
 [-0.03374026  1.          0.0146961 ]
 [-0.01696251  0.0146961   1.        ]]

Running PCA on X (no standardization, so variance dominates)...
Explained variance ratio:
[0.60093347 0.39307216 0.00599437]

Principal component vectors (rows):
  PC1: [ 0.99685796 -0.07918941 -0.00179954]
  PC2: [0.07919236 0.99685802 0.00163255]
  PC3: [ 0.0016646  -0.00176993  0.99999705]

Interpretation:
  If PC1 and PC2 are mostly aligned with X1 and X2,
  their vectors should have large weight in the first two coordinates.

Now running PCR with different numbers of PCs...
Because the target is Y = X1 + X2 + X3, all three features matter.
But PCA 